# Equipetrol Delivery · EC2 en Google Colab: la simulación completa, con GPU y sin GPU

Compila `delivery_sim` **dos veces en esta misma máquina**: con CUDA (`-DUSE_CUDA=ON`: R1 prefiltra las motos más
cercanas en la GPU en cada ronda del despachador y T1 calcula la alcanzabilidad al arrancar) y sin CUDA (todo en la
CPU, el build por defecto). Corre la simulación de la ciudad (4775 nodos, 3000 motos) y la de Equipetrol, 3 veces
cada una, **intercalando** las dos versiones, y mide con `tools/efecto_simulacion.py`: latencia de despacho por
pedido, pedidos asignados por segundo real, entregados por hora simulada, arranque y frames por segundo del bucle de
frames (`--frames`). Las aplicaciones de tipo A (el reporte) no corren durante la simulación: no entran acá.
Tarda unos 25 minutos.

**Antes de correr nada:** *Runtime → Change runtime type → T4 GPU → Save*; después *Runtime → Run all*.

**Al terminar:** descargar `results_simulacion.zip` (lo baja la última celda) y el notebook ejecutado (*File →
Download → Download .ipynb*). **No usar "Guardar en GitHub".**

In [ ]:
# La máquina
!nvidia-smi --query-gpu=name,driver_version,clocks.max.sm,memory.total --format=csv
!nvcc --version | tail -2
!g++ --version | head -1
!lscpu | grep -E 'Model name|^CPU\(s\)|Thread|Core'

In [ ]:
# El código: la rama de EC2 (el hash de src/ identifica lo medido)
BRANCH = "feat/ec2-gpu"
!rm -rf Delivery_Equipetrol && git clone -q -b {BRANCH} https://github.com/OjedaErwin123/Delivery_Equipetrol
!cd Delivery_Equipetrol && git log -1 --format='commit %h  %s' && echo "src/ = $(git rev-parse HEAD:src)"

In [ ]:
# Dos builds, mismas banderas (Release, sin ventana): con CUDA y sin CUDA. Solo delivery_sim y dos pruebas.
!cmake -S Delivery_Equipetrol -B build-gpu -DUSE_CUDA=ON -DDELIVERY_WINDOW=OFF -DCMAKE_BUILD_TYPE=Release 2>&1 | tail -1
!cmake -S Delivery_Equipetrol -B build-cpu -DDELIVERY_WINDOW=OFF -DCMAKE_BUILD_TYPE=Release 2>&1 | tail -1
!cmake --build build-gpu -j2 --target delivery_sim frame_meter_check cli_check 2>&1 | grep -ciE 'warning' ; echo "^ warnings (gpu)"
!cmake --build build-cpu -j2 --target delivery_sim frame_meter_check cli_check 2>&1 | grep -ciE 'warning' ; echo "^ warnings (cpu)"
!./build-gpu/frame_meter_check | tail -1 && ./build-gpu/cli_check Delivery_Equipetrol/config/equipetrol.json | tail -1
!ldd build-cpu/delivery_sim | grep -c cuda ; echo "^ bibliotecas de CUDA en el build sin CUDA (debe ser 0)"

## Las corridas, intercaladas

Cada corrida: `delivery_sim <config> --headless --frames` (60 s reales). Sin ventana, `--frames` arma igual la foto de
cada frame (cada moto con su lock, lo que leería la ventana) sin dibujarla. Se intercalan GPU y CPU para que una
variación de la máquina (Colab es compartida) no caiga toda sobre una versión.

In [ ]:
!rm -rf results && mkdir -p results/ciudad results/equipetrol
for r in range(3):
    !python3 Delivery_Equipetrol/tools/efecto_simulacion.py run build-gpu/delivery_sim Delivery_Equipetrol/config/santa_cruz.json gpu 1 results/ciudad {r}
    !python3 Delivery_Equipetrol/tools/efecto_simulacion.py run build-cpu/delivery_sim Delivery_Equipetrol/config/santa_cruz.json cpu 1 results/ciudad {r}
for r in range(3):
    !python3 Delivery_Equipetrol/tools/efecto_simulacion.py run build-gpu/delivery_sim Delivery_Equipetrol/config/equipetrol.json gpu 1 results/equipetrol {r}
    !python3 Delivery_Equipetrol/tools/efecto_simulacion.py run build-cpu/delivery_sim Delivery_Equipetrol/config/equipetrol.json cpu 1 results/equipetrol {r}

In [ ]:
# Lo que dijo cada versión en stderr (la primera corrida): GPU, tabla de T1, frames
!for f in results/*/*_stderr.txt; do echo "== $f"; cat $f; done

## nsys: el hilo dueño de la GPU durante la simulación

Una corrida con GPU de la ciudad bajo `nsys`: cuántas veces se lanzó cada kernel durante la simulación (R1 en cada
ronda del despachador, T1 una vez al arrancar), cuánto tardan y cuánto se copia. Bajo `nsys` la simulación corre más
lenta: estos números no van a la tabla, solo muestran qué hace la GPU.

In [ ]:
!nsys profile --stats=true --force-overwrite true -o results/simulacion ./build-gpu/delivery_sim Delivery_Equipetrol/config/santa_cruz.json --headless --frames --log results/nsys.log 2>&1 | grep -A14 -E 'cuda_gpu_kern_sum|cuda_gpu_mem_time_sum|cuda_api_sum' | head -60
!rm -f results/nsys.log

## Tablas (desde los CSV)

Mediana [mín–máx] de las corridas. La de la laptop (sin GPU) está en el repo, en `docs/ec2/simulacion/laptop_*.csv`.

In [ ]:
print("Ciudad (4775 nodos, 3000 motos)")
!python3 Delivery_Equipetrol/tools/efecto_simulacion.py table results/ciudad/simulacion.csv
print("Equipetrol (154 nodos, 8 motos)")
!python3 Delivery_Equipetrol/tools/efecto_simulacion.py table results/equipetrol/simulacion.csv
import glob
for f in sorted(glob.glob("Delivery_Equipetrol/docs/ec2/simulacion/laptop_*.csv")):
    print("Laptop:", f)
    !python3 Delivery_Equipetrol/tools/efecto_simulacion.py table {f}

In [ ]:
GUARDAR_EN_DRIVE = False
!cd Delivery_Equipetrol && git rev-parse HEAD:src > ../results/src_tree.txt
!zip -qr results_simulacion.zip results && ls -la results_simulacion.zip
if GUARDAR_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/EC2_resultados && cp results_simulacion.zip /content/drive/MyDrive/EC2_resultados/
from google.colab import files
files.download("results_simulacion.zip")